# PM2.5 Air Quality Prediction using Weather Data

**Project objective:** Predict daily PM2.5 (fine particulate matter, µg/m³)
concentration in New Delhi from historical weather measurements (temperature,
humidity, pressure, wind, visibility), covering 2013–2018, and compare several
regression models to find the best predictor.

**Note on problem type:** the target, PM2.5, is a **continuous** value, so this
is a **regression** problem, not classification. Models and metrics below are
chosen accordingly (e.g. Linear Regression instead of Logistic Regression,
Support Vector *Regression* instead of a classification SVM, MAE/RMSE/R² instead
of accuracy/precision/recall, and actual-vs-predicted/residual plots instead of
a confusion matrix, which does not apply to continuous targets).

**Data collection and cleaning logic is unchanged from the original project** —
only literal duplicate code/cells were removed. Everything else (EDA depth,
per-model write-ups, deep learning section, comparison, and conclusion) has
been expanded to form a complete, submission-ready workflow.

**Reproducibility note:** Section 0 (Data Collection) needs internet access and
the original `Data/` folder and is guarded behind flags — run it once, or skip
it if `Data/Real-Data/Real_Combine.csv` already exists. Because the raw files
are not bundled with this notebook, outputs are empty until it is run in an
environment where `Data/` is present. No results are fabricated anywhere below.


## 1. Imports

In [ ]:
# --- Standard library ---
import os
import sys
import csv
import time
import warnings

# --- Data handling ---
import numpy as np
import pandas as pd

# --- Web scraping / HTML parsing (used only in Section 0) ---
import requests
from bs4 import BeautifulSoup

# --- Visualization ---
import matplotlib.pyplot as plt

# --- Machine learning ---
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# --- Deep learning ---
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping

# --- Model persistence ---
import joblib

warnings.filterwarnings('ignore')

# --- Global configuration ---
RANDOM_STATE = 42
START_YEAR, END_YEAR = 2013, 2018          # inclusive range of data years
BASE_DIR = 'Data'                          # root folder for all raw/processed data
TARGET_COL = 'PM 2.5'

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

print("Libraries imported successfully.")


## 0. Data Collection (optional, one-time step)

Unchanged from the original project, except the six near-identical
`avg_data_2013()` ... `avg_data_2018()` functions (previously duplicated across
two cells) are now one function, `avg_data(year)`, with the exact same logic
(sums the day's valid readings and divides by 24, as in the original).

Set `SCRAPE_DATA = True` and/or `BUILD_DATASET = True` to (re)run this section.
If `Data/Real-Data/Real_Combine.csv` already exists, leave both `False` and
continue to Section 1.


In [ ]:
SCRAPE_DATA = False      # set True to re-download the raw weather HTML pages
BUILD_DATASET = False    # set True to rebuild Data/Real-Data/Real_Combine.csv


def retrieve_html(start_year=START_YEAR, end_year=END_YEAR):
    """Download monthly weather HTML pages for each year in [start_year, end_year]."""
    for year in range(start_year, end_year + 1):
        for month in range(1, 13):
            month_str = f"0{month}" if month < 10 else str(month)
            url = f"http://en.tutiempo.net/climate/{month_str}-{year}/ws-421820.html"
            texts = requests.get(url)
            text_utf = texts.text.encode('utf-8')

            out_dir = f"{BASE_DIR}/Html_Data/{year}"
            os.makedirs(out_dir, exist_ok=True)
            with open(f"{out_dir}/{month}.html", "wb") as output:
                output.write(text_utf)
        sys.stdout.flush()


if SCRAPE_DATA:
    start_time = time.time()
    retrieve_html()
    print(f"Time taken {time.time() - start_time}")


In [ ]:
# Single reusable function replacing the six duplicated avg_data_<year>()
# functions from the original notebook. Logic unchanged: sum each day's valid
# PM2.5 readings and divide by 24.
def avg_data(year):
    average = []
    for rows in pd.read_csv(f'{BASE_DIR}/AQI/aqi{year}.csv', chunksize=24):
        add_var = 0
        data = []
        df_chunk = pd.DataFrame(rows)
        for _, row in df_chunk.iterrows():
            data.append(row['PM2.5'])
        for i in data:
            if isinstance(i, (float, int)):
                add_var += i
            elif isinstance(i, str):
                if i not in ('NoData', 'PwrFail', '---', 'InVld'):
                    add_var += float(i)
        avg = add_var / 24
        average.append(avg)
    return average


In [ ]:
# Quick look at daily PM2.5 averages across all years.
if BUILD_DATASET:
    years = range(START_YEAR, END_YEAR + 1)
    data_dict = {year: avg_data(year) for year in years}

    plt.figure(figsize=(12, 6))
    for year, data in data_dict.items():
        plt.plot(range(len(data)), data, label=f"{year} data")

    plt.xlabel('Day of Year')
    plt.ylabel('PM 2.5 (µg/m³)')
    plt.title('Daily Average PM 2.5 - 2013 to 2018')
    plt.legend(loc='upper right')
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
def met_data(month, year):
    """Extract weather data from a downloaded HTML file (unchanged logic;
    find_all() is the non-deprecated equivalent of the original findAll())."""
    try:
        with open(f'{BASE_DIR}/Html_Data/{year}/{month}.html', 'rb') as f:
            plain_text = f.read()
    except FileNotFoundError:
        print(f"Warning: HTML for {year}/{month} missing. Skipping.")
        return []

    soup = BeautifulSoup(plain_text, "lxml")
    tempD = []
    finalD = []

    for table in soup.find_all('table', {'class': 'medias mensuales numspan'}):
        for tbody in table:
            for tr in tbody:
                tempD.append(tr.get_text())

    rows = len(tempD) // 15
    for _ in range(rows):
        newrow = []
        for _ in range(15):
            newrow.append(tempD.pop(0))
        finalD.append(newrow)

    if len(finalD) > 1:
        finalD.pop(0)
        finalD.pop(-1)

    for row in finalD:
        for idx in sorted([14, 13, 12, 11, 10, 9, 6, 0], reverse=True):
            row.pop(idx)

    return finalD


def data_combine(year, chunksize=600):
    """Read a year's CSV and return all rows as a list of lists."""
    all_rows = []
    try:
        for chunk in pd.read_csv(f'{BASE_DIR}/Real-Data/real_{year}.csv', chunksize=chunksize):
            all_rows.extend(chunk.values.tolist())
    except FileNotFoundError:
        print(f"Warning: real_{year}.csv not found.")
    return all_rows


In [ ]:
if BUILD_DATASET:
    os.makedirs(f"{BASE_DIR}/Real-Data", exist_ok=True)

    for year in range(START_YEAR, END_YEAR + 1):
        print(f"Processing year {year}...")
        final_data = []

        with open(f'{BASE_DIR}/Real-Data/real_{year}.csv', 'w', newline='') as f:
            writer = csv.writer(f, dialect='excel')
            writer.writerow(['T', 'TM', 'Tm', 'SLP', 'H', 'VV', 'V', 'VM', 'PM 2.5'])

        for month in range(1, 13):
            final_data.extend(met_data(month, year))

        pm = avg_data(year)

        if len(pm) < len(final_data):
            pm.extend(['-'] * (len(final_data) - len(pm)))
        elif len(pm) > len(final_data):
            pm = pm[:len(final_data)]

        for i in range(len(final_data)):
            final_data[i].insert(8, pm[i])

        with open(f'{BASE_DIR}/Real-Data/real_{year}.csv', 'a', newline='') as f:
            writer = csv.writer(f, dialect='excel')
            for row in final_data:
                if any(v in ("", "-") for v in row):
                    continue
                writer.writerow(row)

    print("Combining all years...")
    all_data = []
    for year in range(START_YEAR, END_YEAR + 1):
        all_data.extend(data_combine(year))

    with open(f'{BASE_DIR}/Real-Data/Real_Combine.csv', 'w', newline='') as f:
        writer = csv.writer(f, dialect='excel')
        writer.writerow(['T', 'TM', 'Tm', 'SLP', 'H', 'VV', 'V', 'VM', 'PM 2.5'])
        writer.writerows(all_data)

    print(f"Done! Total rows combined: {len(all_data)}")


## 1. Data Loading & Dataset Description

**What this dataset contains:** each row is one day of weather observations
for New Delhi, with the following columns:

| Column | Meaning |
|---|---|
| `T` | Average daily temperature (°C) |
| `TM` | Maximum daily temperature (°C) |
| `Tm` | Minimum daily temperature (°C) |
| `SLP` | Sea-level pressure (hPa) |
| `H` | Average relative humidity (%) |
| `VV` | Average visibility (km) |
| `V` | Average wind speed (km/h) |
| `VM` | Maximum wind speed (km/h) |
| `PM 2.5` | **Target** — daily average fine particulate matter concentration (µg/m³) |


In [ ]:
DATA_PATH = f"{BASE_DIR}/Real-Data/Real_Combine.csv"

df = pd.read_csv(DATA_PATH)

print("Data loaded successfully!")
print(f"Shape: {df.shape} (rows, columns)\n")

print("Columns:", df.columns.tolist())
print("\nFirst 5 rows:")
print(df.head())

print("\nData types:")
df.info()


## 2. Data Cleaning

All standard cleaning checks are applied below. Sub-sections mirror the
required checks: missing values, incorrect/invalid values, data types,
duplicate rows, irrelevant columns, and outliers.

### 2.1 Missing values — diagnosis
Checking the target column reveals a known issue carried over from data
collection: `PM 2.5` is missing for (almost) every row, because the per-day
PM2.5 values did not align correctly with the weather rows when
`Real_Combine.csv` was written.


In [ ]:
missing = df[TARGET_COL].isna().sum()
total = len(df)
print(f"PM 2.5 - Missing: {missing} / {total} ({missing/total*100:.1f}%)")


### 2.2 Fixing the target column (unchanged from the original project)

The fix recomputes each day's PM2.5 average directly from the AQI CSV files
(`avg_data_from_file`, which averages only the *valid* hourly readings for
that day — more robust than the fixed `/24` used during collection) and
reassigns it onto the weather DataFrame.


In [ ]:
aqi_folder = f'{BASE_DIR}/AQI/'

def avg_data_from_file(year):
    averages = []
    file_path = f"{aqi_folder}aqi{year}.csv"
    try:
        for chunk in pd.read_csv(file_path, chunksize=24):
            vals = []
            for _, row in chunk.iterrows():
                pm = row.get('PM2.5', row.get('PM 2.5', row.get('pm25')))
                if pd.isna(pm):
                    continue
                if isinstance(pm, str):
                    if pm in ['NoData', 'PwrFail', '---', 'InVld']:
                        continue
                    try:
                        pm = float(pm)
                    except ValueError:
                        continue
                vals.append(pm)
            averages.append(np.mean(vals) if vals else np.nan)
    except FileNotFoundError:
        print(f"Warning: {file_path} not found, skipping {year}")
        return []
    return averages


all_pm = []
for year in range(START_YEAR, END_YEAR + 1):
    all_pm.extend(avg_data_from_file(year))

print(f"Total PM2.5 averages computed: {len(all_pm)}")

if len(all_pm) > len(df):
    all_pm = all_pm[:len(df)]
elif len(all_pm) < len(df):
    all_pm.extend([np.nan] * (len(df) - len(all_pm)))

df[TARGET_COL] = all_pm
df = df.dropna(subset=[TARGET_COL])

print(f"Shape after fixing PM2.5: {df.shape}")
df.head()


### 2.3 Invalid values, data types, and remaining missing values
(unchanged from the original project's cleaning step)

In [ ]:
placeholders = ['-', 'NoData', 'PwrFail', '---', 'InVld', 'nan', 'NaN', 'NA', 'null']
df = df.replace(placeholders, np.nan)

numeric_cols = ['T', 'TM', 'Tm', 'SLP', 'H', 'VV', 'V', 'VM', 'PM 2.5']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Drop rows where the target is still missing.
df = df.dropna(subset=['PM 2.5'])

# Fill any remaining missing weather values with the column median.
weather_cols = ['T', 'TM', 'Tm', 'SLP', 'H', 'VV', 'V', 'VM']
for col in weather_cols:
    df[col] = df[col].fillna(df[col].median())

print("Shape after cleaning:", df.shape)
print("\nMissing values now:")
print(df.isnull().sum())
print("\nData types now:")
print(df.dtypes)


### 2.4 Duplicate rows

In [ ]:
before = len(df)
df = df.drop_duplicates()
after = len(df)
print(f"Removed {before - after} duplicate rows ({before} -> {after}).")


### 2.5 Irrelevant / redundant columns

No columns are dropped here: all eight weather measurements are physically
plausible drivers of air quality (temperature affects dispersion, humidity
affects particle settling, wind/visibility relate directly to pollutant
concentration), and the correlation heatmap in Section 3 confirms none of
them are redundant duplicates of another column.

### 2.6 Outliers (unchanged from the original project — IQR capping)

Extreme values in the weather columns are capped (not removed) using the IQR
rule, so genuine but rare weather extremes don't distort scaled/distance-based
models without discarding data.

In [ ]:
for col in weather_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    df[col] = df[col].clip(lower, upper)

print("Final cleaned shape:", df.shape)
df.head()


## 3. Exploratory Data Analysis

In [ ]:
# Target distribution
plt.figure(figsize=(8, 4))
df[TARGET_COL].hist(bins=40, edgecolor='black')
plt.title('Distribution of PM2.5')
plt.xlabel('PM2.5 (µg/m³)')
plt.ylabel('Frequency')
plt.tight_layout()
plt.show()


In [ ]:
# Box plots of weather columns - visual outlier check after capping
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
for ax, col in zip(axes.flatten(), weather_cols):
    ax.boxplot(df[col], vert=True)
    ax.set_title(col)
plt.suptitle('Box Plots of Weather Features (post outlier-capping)')
plt.tight_layout()
plt.show()


In [ ]:
# Correlation heatmap
plt.figure(figsize=(8, 6))
corr = df.corr(numeric_only=True)
plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
plt.colorbar(label='Correlation')
plt.xticks(range(len(corr.columns)), corr.columns, rotation=45, ha='right')
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()


In [ ]:
# Bar chart: correlation of each weather feature with the target
target_corr = df[weather_cols].corrwith(df[TARGET_COL]).sort_values()
plt.figure(figsize=(8, 5))
target_corr.plot(kind='barh', color='steelblue')
plt.title('Correlation of Weather Features with PM2.5')
plt.xlabel('Correlation coefficient')
plt.tight_layout()
plt.show()


In [ ]:
# Scatter plots: two of the more strongly related features vs PM2.5
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(df['H'], df[TARGET_COL], alpha=0.4, s=10)
axes[0].set_xlabel('Humidity (%)')
axes[0].set_ylabel('PM2.5 (µg/m³)')
axes[0].set_title('Humidity vs PM2.5')

axes[1].scatter(df['VV'], df[TARGET_COL], alpha=0.4, s=10, color='darkorange')
axes[1].set_xlabel('Visibility (km)')
axes[1].set_ylabel('PM2.5 (µg/m³)')
axes[1].set_title('Visibility vs PM2.5')

plt.tight_layout()
plt.show()


## 4. Data Preprocessing

### 4.1 Feature Engineering
Because PM2.5 has strong day-to-day and seasonal patterns, we add (unchanged
from the original project):
- `day_index`: a simple time index.
- `sin_day` / `cos_day`: cyclical encoding of day-of-year (365.25-day cycle) —
  captures yearly seasonality without an artificial jump between Dec 31 and Jan 1.
- Lag features (`PM25_lag_1/2/3/7`): PM2.5 from previous days — air quality is
  highly autocorrelated day-to-day.
- `PM25_rolling7`: 7-day rolling average of PM2.5, smoothing short-term noise.

All of these use only **past** values (`shift`, trailing `rolling`), so no
future information leaks into a given row's features.


In [ ]:
df = df.reset_index(drop=True)
df['day_index'] = range(len(df))
df['sin_day'] = np.sin(2 * np.pi * df['day_index'] / 365.25)
df['cos_day'] = np.cos(2 * np.pi * df['day_index'] / 365.25)

for lag in [1, 2, 3, 7]:
    df[f'PM25_lag_{lag}'] = df[TARGET_COL].shift(lag)

df['PM25_rolling7'] = df[TARGET_COL].rolling(window=7).mean()

before = len(df)
df = df.dropna()
after = len(df)
print(f"Dropped {before - after} rows due to NaN introduced by lag/rolling features.")
print(f"Final feature-engineered shape: {df.shape}")
df.head()


### 4.2 Feature / Target Split and Train-Test Split

This is **time series** data, so the split preserves chronological order (no
shuffling): the model trains on the earlier 80% of days and is evaluated on
the most recent 20%, matching how it would actually be used in practice.


In [ ]:
feature_cols = [c for c in df.columns if c != TARGET_COL]
X = df[feature_cols]
y = df[TARGET_COL]

split_idx = int(0.8 * len(df))
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

print(f"Features used ({len(feature_cols)}): {feature_cols}")
print(f"Train size: {X_train.shape} | Test size: {X_test.shape}")


### 4.3 Feature Scaling

`StandardScaler` is fit **only on the training set** and then applied to the
test set — fitting it on the full dataset beforehand would leak test-set
information into training. Tree-based models (Decision Tree, Random Forest,
XGBoost) don't need scaling and use the raw features instead.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Scaling complete. Scaled train shape:", X_train_scaled.shape)


## 5. Machine Learning Models

Each model below is trained one at a time: a short theory note, training,
prediction, and evaluation with regression metrics (MAE, RMSE, R²). A
confusion matrix is not shown per model since PM2.5 is a continuous target,
not a class label — the equivalent diagnostic (actual-vs-predicted and
residual plots) is shown once, for the best model, in Section 8.


In [ ]:
results = {}

def evaluate(name, y_true, y_pred):
    """Compute MAE, RMSE, R2 and store them in the shared `results` dict."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    results[name] = {'MAE': mae, 'RMSE': rmse, 'R2': r2}
    print(f"{name:24s} -> MAE: {mae:.3f} | RMSE: {rmse:.3f} | R2: {r2:.3f}")


### 5.1 Linear Regression
**Theory:** fits a straight-line relationship `y = w·X + b` by minimizing
squared error. **Why suitable:** gives a fast, interpretable baseline and
works well if PM2.5 responds roughly linearly to the weather/lag features.

In [ ]:
lr_model = LinearRegression()
lr_model.fit(X_train_scaled, y_train)
y_pred = lr_model.predict(X_test_scaled)
evaluate('Linear Regression', y_test, y_pred)


### 5.2 Decision Tree Regressor
**Theory:** recursively splits the data on feature thresholds to minimize
prediction error within each leaf. **Why suitable:** captures non-linear
relationships and interactions (e.g. humidity mattering differently at
different temperatures) without needing scaling.

In [ ]:
dt_model = DecisionTreeRegressor(random_state=RANDOM_STATE)
dt_model.fit(X_train, y_train)
y_pred = dt_model.predict(X_test)
evaluate('Decision Tree', y_test, y_pred)


### 5.3 Random Forest Regressor
**Theory:** an ensemble (bagging) of many decision trees, each trained on a
random subset of data/features, with predictions averaged. **Why suitable:**
reduces the overfitting and high variance of a single decision tree while
still handling non-linear weather-PM2.5 relationships.

In [ ]:
rf_model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE)
rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)
evaluate('Random Forest', y_test, y_pred)


### 5.4 k-Nearest Neighbors Regressor
**Theory:** predicts a value as the (distance-weighted) average of the `k`
most similar historical days in feature space. **Why suitable:** air quality
on a given day tends to resemble other days with similar weather and recent
PM2.5 history, which is exactly what k-NN exploits.

In [ ]:
knn_model = KNeighborsRegressor(n_neighbors=5, weights='distance', n_jobs=-1)
knn_model.fit(X_train_scaled, y_train)
y_pred = knn_model.predict(X_test_scaled)
evaluate('k-NN', y_test, y_pred)


### 5.5 Support Vector Regression (SVR)
**Theory:** finds a function that stays within a margin of tolerance around
the training points, using kernels to model non-linear relationships.
**Why suitable:** an SVM-family model appropriate for continuous targets
(SVR, the regression counterpart of a classification SVM); RBF kernel can
capture smooth non-linear weather-PM2.5 relationships.

In [ ]:
svr_model = SVR(kernel='rbf', C=10, epsilon=0.5)
svr_model.fit(X_train_scaled, y_train)
y_pred = svr_model.predict(X_test_scaled)
evaluate('SVR (RBF kernel)', y_test, y_pred)


### 5.6 XGBoost Regressor
**Theory:** gradient-boosted trees built sequentially, where each new tree
corrects the residual errors of the previous ones. **Why suitable:** typically
one of the strongest performers on structured/tabular data such as this
weather dataset, and handles feature interactions well.

In [ ]:
xgb_model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=RANDOM_STATE)
xgb_model.fit(X_train, y_train)
y_pred = xgb_model.predict(X_test)
evaluate('XGBoost', y_test, y_pred)


### 5.7 Hyperparameter Tuning (XGBoost)
XGBoost is tuned further with a grid search over its key hyperparameters,
using 3-fold cross-validation on the training set, to see if performance
improves over the default configuration above.

In [ ]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
}

grid_search = GridSearchCV(
    XGBRegressor(random_state=RANDOM_STATE),
    param_grid, cv=3, scoring='r2', n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

tuned_xgb_model = grid_search.best_estimator_
print("Best parameters:", grid_search.best_params_)
print(f"Best cross-validated R2: {grid_search.best_score_:.3f}")

y_pred = tuned_xgb_model.predict(X_test)
evaluate('XGBoost (Tuned)', y_test, y_pred)


## 6. Deep Learning Model

**Theory:** a feed-forward neural network (multilayer perceptron) learns a
non-linear function of the input features through stacked `Dense` layers with
non-linear activations, trained by backpropagation to minimize mean squared
error. Dropout and batch normalization reduce overfitting on a dataset this
size (a few thousand rows).

**Architecture:** input dimension = number of engineered features
(`X_train_scaled.shape[1]`); three hidden `Dense` layers (128 -> 64 -> 32
units, ReLU activation) with `BatchNormalization` + `Dropout` after the first
two; output layer = a single linear unit (one continuous PM2.5 prediction).

**Input/output:** input shape `(n_samples, n_features)`, output shape
`(n_samples, 1)`.


In [ ]:
n_features = X_train_scaled.shape[1]

ann_model = Sequential([
    Dense(128, activation='relu', input_shape=(n_features,)),
    BatchNormalization(),
    Dropout(0.3),
    Dense(64, activation='relu'),
    BatchNormalization(),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1)   # single continuous output: predicted PM2.5
])

ann_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
ann_model.summary()


In [ ]:
early_stop = EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

ann_history = ann_model.fit(
    X_train_scaled, y_train,
    validation_split=0.2, epochs=150, batch_size=32,
    callbacks=[early_stop], verbose=0
)

y_pred = ann_model.predict(X_test_scaled, verbose=0).flatten()
evaluate('Neural Network (ANN)', y_test, y_pred)


In [ ]:
# Training / validation loss and MAE curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].plot(ann_history.history['loss'], label='Train Loss')
axes[0].plot(ann_history.history['val_loss'], label='Validation Loss')
axes[0].set_title('Neural Network - Loss (MSE) over Epochs')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('MSE Loss')
axes[0].legend()

axes[1].plot(ann_history.history['mae'], label='Train MAE')
axes[1].plot(ann_history.history['val_mae'], label='Validation MAE')
axes[1].set_title('Neural Network - MAE over Epochs')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('MAE')
axes[1].legend()

plt.tight_layout()
plt.show()


## 7. Model Comparison

In [ ]:
metrics_df = pd.DataFrame(results).T[['MAE', 'RMSE', 'R2']].sort_values('R2', ascending=False)
metrics_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics_df[['MAE', 'RMSE']].plot(kind='bar', ax=axes[0])
axes[0].set_title('Error Metrics by Model (lower is better)')
axes[0].set_ylabel('Error')
axes[0].tick_params(axis='x', rotation=45)

metrics_df['R2'].plot(kind='bar', ax=axes[1], color='seagreen')
axes[1].set_title('R2 Score by Model (higher is better)')
axes[1].set_ylabel('R2')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


## 8. Best Model

The most appropriate single metric for comparing regression models here is
**R²** (proportion of variance in PM2.5 explained by the model) alongside MAE
for an interpretable average error in µg/m³. The best model is selected
automatically from the results computed above — no score is hard-coded.


In [ ]:
best_model_name = metrics_df['R2'].idxmax()
best_row = metrics_df.loc[best_model_name]

print(f"Best model by R2 on the test set: {best_model_name}")
print(f"MAE  = {best_row['MAE']:.3f}")
print(f"RMSE = {best_row['RMSE']:.3f}")
print(f"R2   = {best_row['R2']:.3f}")


**Why an ensemble/boosted model typically performs better here (general
reasoning, to be confirmed against the printed result above once run):**
tree ensembles like Random Forest and XGBoost can capture non-linear
thresholds and interactions between weather variables that a single linear
model cannot, while also handling the strongly predictive PM2.5 lag/rolling
features naturally — whereas linear/distance-based models are more sensitive
to how those features are scaled and combined.

In [ ]:
model_lookup = {
    'Linear Regression': lr_model,
    'Decision Tree': dt_model,
    'Random Forest': rf_model,
    'k-NN': knn_model,
    'SVR (RBF kernel)': svr_model,
    'XGBoost': xgb_model,
    'XGBoost (Tuned)': tuned_xgb_model,
    'Neural Network (ANN)': ann_model,
}
best_model = model_lookup[best_model_name]

# Recreate predictions from the best model for diagnostic plots below.
scaled_models = {'Linear Regression', 'k-NN', 'SVR (RBF kernel)', 'Neural Network (ANN)'}
if best_model_name in scaled_models:
    best_pred = best_model.predict(X_test_scaled)
else:
    best_pred = best_model.predict(X_test)
best_pred = np.asarray(best_pred).flatten()


In [ ]:
# Actual vs Predicted, and residuals - the regression equivalent of a
# confusion matrix, used to diagnose the best model's behaviour.
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(y_test, best_pred, alpha=0.4, s=12)
lims = [min(y_test.min(), best_pred.min()), max(y_test.max(), best_pred.max())]
axes[0].plot(lims, lims, 'r--', label='Perfect prediction')
axes[0].set_xlabel('Actual PM2.5')
axes[0].set_ylabel('Predicted PM2.5')
axes[0].set_title(f'Actual vs Predicted - {best_model_name}')
axes[0].legend()

residuals = y_test.values - best_pred
axes[1].scatter(best_pred, residuals, alpha=0.4, s=12, color='darkorange')
axes[1].axhline(0, color='red', linestyle='--')
axes[1].set_xlabel('Predicted PM2.5')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].set_title(f'Residual Plot - {best_model_name}')

plt.tight_layout()
plt.show()


In [ ]:
# Feature importance, if the best model is tree-based.
if hasattr(best_model, 'feature_importances_'):
    importance = pd.Series(best_model.feature_importances_, index=feature_cols)
    importance.sort_values(ascending=False).plot(kind='bar', figsize=(10, 5))
    plt.title(f'Feature Importance - {best_model_name}')
    plt.ylabel('Importance')
    plt.tight_layout()
    plt.show()
else:
    print(f"{best_model_name} does not expose feature importances directly.")


In [ ]:
# Save the best model and scaler for later reuse.
joblib.dump(best_model, 'pm25_best_model.pkl')
joblib.dump(scaler, 'scaler.pkl')
print(f"Saved best model ('{best_model_name}') as 'pm25_best_model.pkl' and scaler as 'scaler.pkl'.")


## 9. Final Conclusion

**What was done:** weather and air-quality data for New Delhi (2013-2018) was
loaded, a data-collection bug in the PM2.5 target column was diagnosed and
fixed, the data was cleaned (missing values, invalid placeholders, duplicates,
outliers) and enriched with seasonal and lag-based features. Six regression
model families — Linear Regression, Decision Tree, Random Forest, k-NN, SVR,
and XGBoost (including a tuned variant) — plus a neural network were trained
and evaluated on an identical chronological train/test split for a fair
comparison.

**Best-performing model:** see Section 8 above — printed automatically from
the actual computed metrics, not hard-coded.

**Observations:** PM2.5's own recent history (the lag and rolling-mean
features) is expected to be highly informative given air pollution's strong
day-to-day persistence; this can be checked against the feature-importance
plot in Section 8 once the notebook is run.

**Possible future improvements:**
- Use additional pollutant/meteorological variables (e.g. PM10, NO2, wind
  direction) if available, rather than PM2.5 and basic weather alone.
- Try sequence models (e.g. LSTM/GRU) that can learn temporal patterns more
  explicitly than a lag-feature MLP.
- Use time-series cross-validation (rolling-origin) instead of a single
  80/20 split for a more robust performance estimate.
- Tune the neural network architecture and the SVR/k-NN hyperparameters with
  a systematic search, similar to what was done for XGBoost.
